In [ ]:
from pandas import read_csv

In [ ]:
%pip install json5

In [ ]:
!wget -P data/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_1.csv
!wget -P data/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_2.csv
!wget -P data/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_3.csv

In [ ]:
!wget -P scripts/ https://raw.githubusercontent.com/saksham-joshi/Senti-Vox-CB-Model/refs/heads/main/scripts/tokenizer.py

In [ ]:
!wget -P scripts/ https://raw.githubusercontent.com/saksham-joshi/Senti-Vox-CB-Model/refs/heads/main/scripts/values.json

In [ ]:
!ls -a

In [ ]:
df1 = read_csv("data/goemotions_1.csv")
df2 = read_csv("data/goemotions_2.csv")
df3 = read_csv("data/goemotions_3.csv")

In [ ]:
df1.head()

In [ ]:
df2.head()

In [ ]:
df3.head()

In [ ]:
print(len(df1))
print(len(df2))
print(len(df3))

In [ ]:
df1.tail()

In [ ]:
df2.tail()

In [ ]:
df3.tail()

In [ ]:
df1.columns

In [ ]:
columns_with_emotions = ['admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise', 'neutral']

len(columns_with_emotions)

In [ ]:
EMOTION_TO_SENTIMENT = {
    # Positive emotions
    "admiration":  "pos",
    "amusement":   "pos",
    "approval":    "pos",
    "caring":      "pos",
    "desire":      "pos",
    "excitement":  "pos",
    "gratitude":   "pos",
    "joy":         "pos",
    "love":        "pos",
    "optimism":    "pos",
    "pride":       "pos",
    "relief":      "pos",
    # Negative emotions
    "anger":          "neg",
    "annoyance":      "neg",
    "disappointment": "neg",
    "disapproval":    "neg",
    "disgust":        "neg",
    "embarrassment":  "neg",
    "fear":           "neg",
    "grief":          "neg",
    "nervousness":    "neg",
    "remorse":        "neg",
    "sadness":        "neg",
    # Ambiguous emotions (can be positive or negative depending on context)
    "confusion":   "amb",
    "curiosity":   "amb",
    "realization": "amb",
    "surprise":    "amb",
    # Neutral
    "neutral":     "neu",
}

In [ ]:
json_input = []
from scripts.tokenizer import clean_text

for i in (df1, df2, df3) :
    for j in range(len(i)) :
        txt = clean_text(i.iloc[j]["text"]).strip()

        if txt.__len__() == 0 : continue

        dt = {
            "text" : txt,
            "senti" : {
                "pos" : 0,
                "neg" : 0,
                "neu" : 0,
                "amb" : 0
            }
        }
        for column in columns_with_emotions :
            if int(i.iloc[j][column]) == 1 : dt["senti"][EMOTION_TO_SENTIMENT[column]] += 1
        json_input.append(dt)

In [ ]:
print(len(json_input))
print(len(json_input) == len(df1) + len(df2) + len(df3))


In [ ]:
from json5 import dump

with open("go-sentiment.json", "w", encoding="utf-8") as file:
    dump(json_input, file, indent=2, ensure_ascii=False, trailing_commas=False, allow_nan=False, quote_keys=True)

In [ ]:
!ls

In [ ]:
from huggingface_hub import login, upload_file

# (optional) Login with your Hugging Face credentials
login()

upload_file(
    path_or_fileobj="go-sentiment.json",
    path_in_repo="go-sentiment.json",
    repo_id="saksham-joshi/Go-Sentiment",
    repo_type="dataset"
)